In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("../data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
ground_truth[10]

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [9]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [10]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [ ]:
#This is one pair of generated question <-> document (faq)
q = ground_truth[10]
q

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [ ]:
#And this is the cannonical question for the document above.
#In the next steps, we need to compare the generated with the canonical, which we do throgh LLM as judge. 
doc_idx[q['document']]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

In [13]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [14]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
    course='llm-zoomcamp',
)

In [15]:
q['question']

'How do I join the Office Hours or live workshop if I don’t have the Zoom link?'

In [16]:
answer = assistant.rag(q['question'])

In [19]:
print(answer)

The Zoom link is only published to instructors/presenters/TAs.

As a student, you can join via:
- YouTube Live on the DataTalksClub channel
- The video URL posted in the announcements channel on Telegram and Slack before the session starts

For questions, use Slido during the live session rather than posting in chat.


In [18]:
assistant.total_cost()

0.00078075

In [20]:
print(answer)

The Zoom link is only published to instructors/presenters/TAs.

As a student, you can join via:
- YouTube Live on the DataTalksClub channel
- The video URL posted in the announcements channel on Telegram and Slack before the session starts

For questions, use Slido during the live session rather than posting in chat.


In [21]:
doc_id = q["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'

In [22]:
rag_result = {
    "question": q['question'],
    "answer_llm": answer,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'answer_llm': 'The Zoom link is only published to instructors/presenters/TAs.\n\nAs a student, you can join via:\n- YouTube Live on the DataTalksClub channel\n- The video URL posted in the announcements channel on Telegram and Slack before the session starts\n\nFor questions, use Slido during the live session rather than posting in chat.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.',
 'document': '489dd1c9d9'}

In [23]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [24]:
record = generate_rag_answer(q)
record

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'answer_llm': 'You can join the Office Hours or live workshop without the Zoom link by watching via **YouTube Live**. The **video URL is posted in the announcements channel on Telegram and Slack before the session starts**, and it’s also available on the **DataTalksClub YouTube channel**.\n\nFor questions during the live session, use **Slido** rather than chat, since chat messages may be missed.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active

In [25]:
assistant.total_cost()

0.0016155

In [26]:
assistant.reset_usage()

In [27]:
assistant.total_cost()

0.0

In [28]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
#This piece of code is not working. 
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/395 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [31]:
results[:10]

NameError: name 'results' is not defined

In [27]:
df_results = pd.DataFrame(results)

In [28]:
df_results.head()

,question,answer_llm,answer_orig,document
0,Is it okay to join the course late if I just f...,"Yes, you can still join the course late. If yo...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,Can I still take this course even if I missed ...,"Yes, you can still join if you missed the star...","Yes, but if you want to receive a certificate,...",74eb249bbf
2,If I join after the course has already started...,"Yes, as long as you join while the course is s...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,Do I need to submit my project before submissi...,"Yes — to get the certificate, you need to subm...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,I’m a bit late to the course—what do I need to...,"To still earn the certificate, you need to:\n\...","Yes, but if you want to receive a certificate,...",74eb249bbf


In [29]:
assistant.total_cost()


0.34332825

In [ ]:
df_results.to_csv("data/rag-answers-new.csv", index=False)